
# S&P 500 Shared Datasets Pipeline

This notebook rebuilds the shared datasets layer for the S&P 500 project family.

Outputs written by this notebook:
- `datasets/exports/annual-returns.json`
- `datasets/exports/sp500-market-map.json`
- `datasets/exports/sp500-sector-returns.json`
- `datasets/exports/sp500-bubbles.json`

Design goal:
- keep data generation in one place
- keep apps as consumers of shared exports
- explain clearly what each cell downloads and what each cell only reads from local manual sources



## Cell 1: Setup and project paths

This cell does **not download anything**.

It:
- imports the Python modules used in the notebook
- detects the repository root dynamically
- defines the shared `datasets/` directories
- defines the source and export paths used later


In [ ]:

from __future__ import annotations

import csv
import json
import math
import re
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from statistics import mean
from typing import Any
from urllib.request import Request, urlopen


def detect_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not detect market-map project root from the current working directory.')


ROOT = detect_root()
DATASETS_DIR = ROOT / 'datasets'
SOURCES_DIR = DATASETS_DIR / 'sources'
INTERMEDIATE_DIR = DATASETS_DIR / 'intermediate'
EXPORTS_DIR = DATASETS_DIR / 'exports'

for directory in [DATASETS_DIR, SOURCES_DIR, INTERMEDIATE_DIR, EXPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MARKET_MAP_CSV = ROOT / 'sources' / 'manual' / 'notebook' / 'sp500_market_map_returns.csv'
TIMESERIES_WIDE_CSV = ROOT / 'sources' / 'manual' / 'notebook' / 'sp500_timeseries_15y_wide.csv'
VC_RAW_DIR = ROOT / 'composition' / 'data' / 'raw'

ANNUAL_EXPORT = EXPORTS_DIR / 'annual-returns.json'
MARKET_MAP_EXPORT = EXPORTS_DIR / 'sp500-market-map.json'
SECTOR_RETURNS_EXPORT = EXPORTS_DIR / 'sp500-sector-returns.json'
BUBBLES_EXPORT = EXPORTS_DIR / 'sp500-bubbles.json'

print('ROOT =', ROOT)
print('DATASETS_DIR =', DATASETS_DIR)
print('EXPORTS_DIR =', EXPORTS_DIR)



## Cell 2: Shared helpers

This cell does **not download anything**.

It defines reusable helpers for:
- HTTP downloads
- numeric parsing
- quantiles and regressions
- common formatting
- sector configuration used by multiple datasets


In [ ]:

SECTOR_CONFIG = {
    'Information Technology': {'key': 'information-technology', 'es': 'Tecnologia', 'color': '#78b9ff'},
    'Financials': {'key': 'financials', 'es': 'Financieras', 'color': '#22c55e'},
    'Communication Services': {'key': 'communication-services', 'es': 'Servicios de comunicación', 'color': '#b793ff'},
    'Consumer Discretionary': {'key': 'consumer-discretionary', 'es': 'Consumo discrecional', 'color': '#8395ff'},
    'Health Care': {'key': 'health-care', 'es': 'Salud', 'color': '#e7e3ff'},
    'Industrials': {'key': 'industrials', 'es': 'Industriales', 'color': '#ff9cbc'},
    'Consumer Staples': {'key': 'consumer-staples', 'es': 'Consumo básico', 'color': '#ff985f'},
    'Energy': {'key': 'energy', 'es': 'Energía', 'color': '#f4c84d'},
    'Utilities': {'key': 'utilities', 'es': 'Utilities', 'color': '#d8df53'},
    'Materials': {'key': 'materials', 'es': 'Materiales', 'color': '#ff6767'},
    'Real Estate': {'key': 'real-estate', 'es': 'Real Estate', 'color': '#b7ece0'},
}


def download_text(url: str) -> str:
    req = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urlopen(req) as response:
        return response.read().decode('utf-8')


def parse_number(value: Any) -> float:
    if value is None:
        return float('nan')
    text = str(value).strip().replace('$', '').replace('%', '').replace(',', '')
    if not text:
        return float('nan')
    try:
        return float(text)
    except ValueError:
        return float('nan')


def clean_company_name(name: str) -> str:
    return (
        str(name)
        .replace(' (The)', '')
        .replace(' (Class A)', ' A')
        .replace(' (Class B)', ' B')
        .replace(' (Class C)', ' C')
        .replace(' Corporation', '')
        .strip()
    )


def quantile(sorted_values: list[float], q: float) -> float:
    if not sorted_values:
        return 0.0
    index = (len(sorted_values) - 1) * q
    lo = math.floor(index)
    hi = math.ceil(index)
    if lo == hi:
        return sorted_values[lo]
    w = index - lo
    return sorted_values[lo] * (1 - w) + sorted_values[hi] * w


def sample_std(values: list[float]) -> float | None:
    if len(values) < 2:
        return None
    mu = sum(values) / len(values)
    variance = sum((v - mu) ** 2 for v in values) / (len(values) - 1)
    return math.sqrt(variance)


def solve_linear_system(matrix: list[list[float]], rhs: list[float]) -> list[float] | None:
    n = len(rhs)
    a = [row[:] + [rhs[i]] for i, row in enumerate(matrix)]
    for col in range(n):
        pivot = max(range(col, n), key=lambda row: abs(a[row][col]))
        if abs(a[pivot][col]) < 1e-10:
            return None
        a[col], a[pivot] = a[pivot], a[col]
        div = a[col][col]
        for j in range(col, n + 1):
            a[col][j] /= div
        for row in range(n):
            if row == col:
                continue
            factor = a[row][col]
            for j in range(col, n + 1):
                a[row][j] -= factor * a[col][j]
    return [row[n] for row in a]


def fit_simple_regression(samples: list[dict[str, float]]) -> dict[str, float] | None:
    if len(samples) < 2:
        return None
    mean_x = sum(s['x'] for s in samples) / len(samples)
    mean_y = sum(s['y'] for s in samples) / len(samples)
    num = sum((s['x'] - mean_x) * (s['y'] - mean_y) for s in samples)
    den = sum((s['x'] - mean_x) ** 2 for s in samples)
    if den == 0:
        return None
    slope = num / den
    intercept = mean_y - slope * mean_x
    return {'slope': slope, 'intercept': intercept}


def fit_ridge_regression(samples: list[dict[str, Any]], lmbd: float = 1.0) -> dict[str, Any] | None:
    if len(samples) < 10:
        return None
    feature_count = len(samples[0]['x'])
    means = [0.0] * feature_count
    stds = [0.0] * feature_count
    for s in samples:
        for i, value in enumerate(s['x']):
            means[i] += value
    means = [value / len(samples) for value in means]
    for s in samples:
        for i, value in enumerate(s['x']):
            stds[i] += (value - means[i]) ** 2
    stds = [math.sqrt(value / max(len(samples) - 1, 1)) or 1.0 for value in stds]
    z_samples = [
        {'x': [(value - means[i]) / stds[i] for i, value in enumerate(s['x'])], 'y': s['y']}
        for s in samples
    ]
    y_mean = sum(s['y'] for s in z_samples) / len(z_samples)
    xtx = [[0.0] * feature_count for _ in range(feature_count)]
    xty = [0.0] * feature_count
    for s in z_samples:
        for i in range(feature_count):
            xty[i] += s['x'][i] * (s['y'] - y_mean)
            for j in range(feature_count):
                xtx[i][j] += s['x'][i] * s['x'][j]
    for i in range(feature_count):
        xtx[i][i] += lmbd
    betas = solve_linear_system(xtx, xty)
    if betas is None:
        return None
    return {'means': means, 'stds': stds, 'betas': betas, 'y_mean': y_mean}


def predict_ridge(model: dict[str, Any], x: list[float]) -> float:
    z = [(value - model['means'][i]) / model['stds'][i] for i, value in enumerate(x)]
    prediction = model['y_mean']
    for i, value in enumerate(z):
        prediction += model['betas'][i] * value
    return prediction



## Cell 3: Annual returns dataset

This section **downloads** three remote sources:
- Official Data: monthly S&P 500 total-return history
- FRED `SP500`: daily index level history
- FRED `VIXCLS`: daily VIX closes

The goal is to build the shared dataset used by:
- `histogram`
- `risk-indicator`
- `index-forecast`

The output is written to `datasets/exports/annual-returns.json`.


In [ ]:

SOURCE_URL = 'https://www.officialdata.org/us/stocks/s-p-500/1871'
SP500_DAILY_SOURCE_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=SP500'
VIX_SOURCE_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=VIXCLS'
BUCKETS = [-50, -40, -30, -20, -10, 0, 10, 20, 30, 40, 50]


def clamp(value: float, min_value: float, max_value: float) -> float:
    return max(min_value, min(max_value, value))


def bucket_for(return_pct: float) -> int:
    if return_pct >= 50:
        return 50
    return int(clamp(math.floor(return_pct / 10) * 10, -50, 50))


def parse_official_data_html(html: str) -> list[dict[str, Any]]:
    entries = []
    seen = set()
    pattern = re.compile(r'\["(\d{4})-(\d{2})",([0-9.]+),(-?[0-9.eE]+)(?:,([0-9.]+))?\]')
    for year, month, amount, month_return, price_level in pattern.findall(html):
        key = f"{year}-{month}"
        if key in seen:
            continue
        seen.add(key)
        entries.append({
            'year': int(year),
            'month': int(month),
            'key': key,
            'amount': float(amount),
            'monthReturnPct': float(month_return) * 100,
            'priceLevel': float(price_level) if price_level else None,
        })
    return sorted(entries, key=lambda row: row['key'])


def parse_fred_level_csv(csv_text: str, value_name: str) -> list[dict[str, Any]]:
    rows = []
    for line in csv_text.strip().splitlines()[1:]:
        date, value = line.split(',')
        if value == '.':
            continue
        rows.append({
            'date': date,
            'year': int(date[:4]),
            'month': int(date[5:7]),
            'key': f"{date[:4]}-{date[5:7]}",
            value_name: float(value),
        })
    return rows


def build_annual_returns(monthly: list[dict[str, Any]]) -> list[dict[str, Any]]:
    december = [row for row in monthly if row['month'] == 12]
    annual = []
    for prev, current in zip(december[:-1], december[1:]):
        return_pct = ((current['amount'] / prev['amount']) - 1) * 100
        annual.append({'year': current['year'], 'returnPct': return_pct, 'bucket': bucket_for(return_pct)})
    return annual


def build_annual_volatility(monthly: list[dict[str, Any]]) -> list[dict[str, Any]]:
    years = sorted({row['year'] for row in monthly})
    annual_risk = []
    for year in years:
        entries = sorted((row for row in monthly if row['year'] == year), key=lambda row: row['month'])
        monthly_returns = [row['monthReturnPct'] / 100 for row in entries if math.isfinite(row['monthReturnPct'])]
        std = sample_std(monthly_returns)
        if std is None:
            continue
        annual_risk.append({
            'year': year,
            'volatilityPct': std * math.sqrt(12) * 100,
            'annualized': False,
            'monthsElapsed': len(entries),
        })
    return annual_risk


def build_annual_vix(vix_daily: list[dict[str, Any]]) -> list[dict[str, Any]]:
    grouped = defaultdict(list)
    for row in vix_daily:
        grouped[row['year']].append(row['vixClose'])
    return [{'year': year, 'vixAvg': sum(values) / len(values)} for year, values in sorted(grouped.items())]


def build_monthly_vix(vix_daily: list[dict[str, Any]]) -> list[dict[str, Any]]:
    grouped = defaultdict(list)
    for row in vix_daily:
        grouped[row['key']].append(row['vixClose'])
    rows = []
    for key, values in sorted(grouped.items()):
        year, month = key.split('-')
        rows.append({'year': int(year), 'month': int(month), 'key': key, 'vixAvg': sum(values) / len(values)})
    return rows


def build_daily_vix_stats(vix_daily: list[dict[str, Any]], window_size: int = 50) -> list[dict[str, Any]]:
    series = []
    for i in range(window_size - 1, len(vix_daily)):
        window = vix_daily[i - window_size + 1 : i + 1]
        values = [row['vixClose'] for row in window]
        ma = sum(values) / len(values)
        std = sample_std(values)
        if not std or not math.isfinite(std):
            continue
        current = vix_daily[i]
        score = (current['vixClose'] - ma) / std
        deviation_pct = ((current['vixClose'] / ma) - 1) * 100
        series.append({
            'date': current['date'],
            'year': current['year'],
            'month': current['month'],
            'key': current['key'],
            'vixClose': round(current['vixClose'], 2),
            'ma50': round(ma, 2),
            'std50': round(std, 2),
            'score': round(score, 2),
            'deviationPct': round(deviation_pct, 2),
        })
    return series


def build_risk_indicator(vix_daily: list[dict[str, Any]], start_date: str, end_date: str) -> dict[str, Any]:
    series = [row for row in build_daily_vix_stats(vix_daily) if start_date <= row['date'] <= end_date]
    scores = sorted(row['score'] for row in series)
    fear_threshold = round(quantile(scores, 0.9), 2)
    complacency_threshold = round(quantile(scores, 0.1), 2)
    markers = []
    for index, row in enumerate(series):
        if index == 0 or index == len(series) - 1:
            continue
        prev_row = series[index - 1]
        next_row = series[index + 1]
        is_peak = row['score'] >= fear_threshold and row['score'] >= prev_row['score'] and row['score'] >= next_row['score']
        is_trough = row['score'] <= complacency_threshold and row['score'] <= prev_row['score'] and row['score'] <= next_row['score']
        if is_peak or is_trough:
            markers.append(row)
    return {
        'startDate': start_date,
        'endDate': end_date,
        'fearThreshold': fear_threshold,
        'complacencyThreshold': complacency_threshold,
        'series': series,
        'markers': markers,
        'latest': series[-1] if series else None,
    }


def build_annual_forecast_chart(vix_daily: list[dict[str, Any]], annual_return_entries: list[dict[str, Any]]) -> dict[str, Any]:
    daily_stats = build_daily_vix_stats(vix_daily)
    year_end_stats = {}
    for year in sorted({row['year'] for row in daily_stats}):
        stats = [row for row in daily_stats if row['year'] == year]
        year_end_stats[str(year)] = stats[-1] if stats else None
    actual_by_year = {str(row['year']): round(row['returnPct'], 2) for row in annual_return_entries}
    years = [year for year in sorted(int(y) for y in year_end_stats.keys()) if 1990 <= year <= 2025]
    forecasts = []
    for issue_year in years:
        training = [
            {'x': year_end_stats[str(year)]['score'], 'y': actual_by_year[str(year + 1)]}
            for year in years
            if year < issue_year and year_end_stats.get(str(year)) and actual_by_year.get(str(year + 1)) is not None
        ]
        if len(training) < 8:
            continue
        model = fit_simple_regression(training)
        feature = year_end_stats.get(str(issue_year))
        if not model or not feature:
            continue
        target_year = issue_year + 1
        forecast_return_pct = model['intercept'] + model['slope'] * feature['score']
        forecasts.append({
            'issueYear': issue_year,
            'targetYear': target_year,
            'issueDate': feature['date'],
            'issueScore': feature['score'],
            'issueVixClose': feature['vixClose'],
            'forecastReturnPct': round(forecast_return_pct, 2),
            'actualReturnPct': actual_by_year.get(str(target_year)),
            'trainingCount': len(training),
        })
    actual_series = [
        {'year': row['year'], 'returnPct': round(row['returnPct'], 2)}
        for row in annual_return_entries if 1991 <= row['year'] <= 2025
    ]
    errors = [abs(row['forecastReturnPct'] - row['actualReturnPct']) for row in forecasts if row['actualReturnPct'] is not None]
    average_abs_error = round(sum(errors) / len(errors), 2) if errors else None
    return {
        'actualSeries': actual_series,
        'forecasts': forecasts,
        'averageAbsError': average_abs_error,
        'latestForecast': forecasts[-1] if forecasts else None,
    }


def build_index_forecast_chart(sp500_daily: list[dict[str, Any]], vix_daily: list[dict[str, Any]]) -> dict[str, Any]:
    horizon_trading_days = 252
    daily_stats_by_date = {row['date']: row for row in build_daily_vix_stats(vix_daily)}
    aligned = []
    for row in sp500_daily:
        stat = daily_stats_by_date.get(row['date'])
        if not stat:
            continue
        aligned.append({
            'date': row['date'],
            'year': row['year'],
            'month': row['month'],
            'level': row['level'],
            'score': stat['score'],
            'vixClose': stat['vixClose'],
        })

    daily_log_returns = [0.0]
    for i in range(1, len(aligned)):
        daily_log_returns.append(math.log(aligned[i]['level'] / aligned[i - 1]['level']))

    def rolling_mean(index: int, window: int) -> float | None:
        start = index - window + 1
        if start < 0:
            return None
        return sum(aligned[i]['level'] for i in range(start, index + 1)) / window

    def rolling_vol(index: int, window: int) -> float | None:
        start = index - window + 1
        if start < 1:
            return None
        values = daily_log_returns[start:index + 1]
        std = sample_std(values)
        return None if std is None else std * math.sqrt(252)

    def trailing_log_return(index: int, window: int) -> float | None:
        start = index - window
        if start < 0:
            return None
        return math.log(aligned[index]['level'] / aligned[start]['level'])

    max_lag = 252

    def build_feature_row(index: int) -> list[float] | None:
        if index < max_lag:
            return None
        ret21 = trailing_log_return(index, 21)
        ret63 = trailing_log_return(index, 63)
        ret126 = trailing_log_return(index, 126)
        ret252 = trailing_log_return(index, 252)
        ma50 = rolling_mean(index, 50)
        ma200 = rolling_mean(index, 200)
        rv21 = rolling_vol(index, 21)
        rv63 = rolling_vol(index, 63)
        values = [ret21, ret63, ret126, ret252, ma50, ma200, rv21, rv63]
        if any(v is None or not math.isfinite(v) for v in values):
            return None
        return [
            aligned[index]['score'],
            math.log(aligned[index]['vixClose']),
            ret21,
            ret63,
            ret126,
            ret252,
            math.log(aligned[index]['level'] / ma50),
            math.log(aligned[index]['level'] / ma200),
            rv21,
            rv63,
        ]

    def fit_forward_model(end_index: int, horizon: int):
        training = []
        for sample_index in range(max_lag, end_index - horizon + 1):
            current = aligned[sample_index]
            future = aligned[sample_index + horizon]
            x = build_feature_row(sample_index)
            if x is None:
                continue
            training.append({'x': x, 'y': math.log(future['level'] / current['level'])})
        if len(training) < 252:
            return None
        model = fit_ridge_regression(training, 2)
        if model is None:
            return None
        historical_mean = sum(sample['y'] for sample in training) / len(training)
        return {'model': model, 'historicalMean': historical_mean, 'trainingCount': len(training)}

    backtest = []
    min_lookback = 252
    for issue_index in range(max_lag + min_lookback, len(aligned) - horizon_trading_days):
        fit = fit_forward_model(issue_index, horizon_trading_days)
        if fit is None:
            continue
        issue = aligned[issue_index]
        target = aligned[issue_index + horizon_trading_days]
        x = build_feature_row(issue_index)
        if x is None:
            continue
        model_return = predict_ridge(fit['model'], x)
        forecast_log_return = clamp(0.5 * model_return + 0.5 * fit['historicalMean'], -0.4, 0.4)
        forecast_level = issue['level'] * math.exp(forecast_log_return)
        backtest.append({
            'issueDate': issue['date'],
            'targetDate': target['date'],
            'issueLevel': round(issue['level'], 2),
            'targetLevel': round(target['level'], 2),
            'forecastLevel': round(forecast_level, 2),
            'issueScore': issue['score'],
            'issueVixClose': issue['vixClose'],
            'forecastReturnPct': round(((forecast_level / issue['level']) - 1) * 100, 2),
            'actualReturnPct': round(((target['level'] / issue['level']) - 1) * 100, 2),
            'trainingCount': fit['trainingCount'],
        })

    latest_forecast = None
    latest_issue_index = len(aligned) - 1
    latest_issue = aligned[latest_issue_index] if aligned else None
    latest_fit = fit_forward_model(len(aligned) - 1, horizon_trading_days) if aligned else None
    if latest_fit and latest_issue:
        x = build_feature_row(latest_issue_index)
        if x is not None:
            model_return = predict_ridge(latest_fit['model'], x)
            forecast_log_return = clamp(0.5 * model_return + 0.5 * latest_fit['historicalMean'], -0.4, 0.4)
            forecast_level = latest_issue['level'] * math.exp(forecast_log_return)
            latest_forecast = {
                'issueDate': latest_issue['date'],
                'targetDate': latest_issue['date'],
                'issueLevel': round(latest_issue['level'], 2),
                'forecastLevel': round(forecast_level, 2),
                'issueScore': latest_issue['score'],
                'issueVixClose': latest_issue['vixClose'],
                'forecastReturnPct': round(((forecast_level / latest_issue['level']) - 1) * 100, 2),
                'trainingCount': latest_fit['trainingCount'],
            }

    errors = [abs(((row['forecastLevel'] - row['targetLevel']) / row['targetLevel']) * 100) for row in backtest]
    average_abs_error = round(sum(errors) / len(errors), 2) if errors else None

    return {
        'horizonTradingDays': horizon_trading_days,
        'actualSeries': [
            {'date': row['date'], 'level': round(row['level'], 2)}
            for row in aligned if row['date'] >= '2017-01-01'
        ],
        'backtestSeries': [row for row in backtest if row['targetDate'] >= '2017-01-01'],
        'averageAbsError': average_abs_error,
        'latestForecast': latest_forecast,
    }


def group_columns(annual: list[dict[str, Any]]) -> list[dict[str, Any]]:
    columns = []
    for bucket in BUCKETS:
        items = sorted(
            [row for row in annual if row['bucket'] == bucket],
            key=lambda row: row['returnPct'],
            reverse=True,
        )
        columns.append({
            'bucket': bucket,
            'items': [
                {
                    'year': row['year'],
                    'returnPct': round(row['returnPct'], 2),
                    'annualized': bool(row.get('annualized', False)),
                    'monthsElapsed': row.get('monthsElapsed'),
                }
                for row in items
            ],
        })
    return columns


official_html = download_text(SOURCE_URL)
sp500_daily_csv = download_text(SP500_DAILY_SOURCE_URL)
vix_csv = download_text(VIX_SOURCE_URL)

monthly = parse_official_data_html(official_html)
sp500_daily = parse_fred_level_csv(sp500_daily_csv, 'level')
vix_daily = parse_fred_level_csv(vix_csv, 'vixClose')

annual = build_annual_returns(monthly)
annual_volatility = build_annual_volatility(monthly)
annual_vix = build_annual_vix(vix_daily)
monthly_vix = build_monthly_vix(vix_daily)
risk_indicator = build_risk_indicator(vix_daily, '2017-01-01', '2025-12-31')
forecast_chart = build_annual_forecast_chart(vix_daily, annual)
index_forecast_chart = build_index_forecast_chart(sp500_daily, vix_daily)

full_range = [row for row in annual if 1875 <= row['year'] <= 2025]
full_volatility_range = [row for row in annual_volatility if 1875 <= row['year'] <= 2025]
full_vix_range = [row for row in annual_vix if 1990 <= row['year'] <= 2025]
full_monthly_range = [row for row in monthly if 1990 <= row['year'] <= 2025]
columns = group_columns(full_range)

return_by_year = {str(row['year']): {'returnPct': round(row['returnPct'], 2), 'bucket': row['bucket']} for row in full_range}
volatility_by_year = {str(row['year']): {'volatilityPct': round(row['volatilityPct'], 2)} for row in full_volatility_range}
vix_by_year = {str(row['year']): {'vixAvg': round(row['vixAvg'], 2)} for row in full_vix_range}
vix_by_month = {row['key']: {'vixAvg': round(row['vixAvg'], 2)} for row in monthly_vix if 1990 <= row['year'] <= 2025}

monthly_scatter = []
for row in full_monthly_range:
    risk = vix_by_month.get(row['key'])
    if not risk:
        continue
    monthly_scatter.append({
        'year': row['year'],
        'month': row['month'],
        'key': row['key'],
        'label': row['key'],
        'returnPct': round(row['monthReturnPct'], 2),
        'vixAvg': risk['vixAvg'],
        'annualized': False,
    })

latest_monthly_point = monthly_scatter[-1] if monthly_scatter else None

annual_payload = {
    'generatedAt': __import__('datetime').datetime.utcnow().isoformat() + 'Z',
    'sourceName': 'Official Data + FRED SP500 + FRED VIXCLS',
    'sourceUrl': f'{SOURCE_URL} | {SP500_DAILY_SOURCE_URL} | {VIX_SOURCE_URL}',
    'methodology': "Annual return is computed from December-to-December changes in the source's cumulative S&P 500 total-return value series. The risk indicator uses daily VIX close from FRED series VIXCLS and calculates a 50-day moving-average deviation score as (VIX - MA50) / rolling 50-day standard deviation. Fear and complacency bands are the 90th and 10th percentiles of the score over the displayed window.",
    'startYear': 1875,
    'endYear': 2025,
    'yearCount': len(full_range),
    'latestYear': 2025,
    'latestReturnPct': return_by_year.get('2025', {}).get('returnPct'),
    'latestAnnualized': False,
    'latestMonthsElapsed': None,
    'previousYear': 2024,
    'previousReturnPct': return_by_year.get('2024', {}).get('returnPct'),
    'latestVolatilityPct': volatility_by_year.get('2025', {}).get('volatilityPct'),
    'previousVolatilityPct': volatility_by_year.get('2024', {}).get('volatilityPct'),
    'latestVixAvg': vix_by_year.get('2025', {}).get('vixAvg'),
    'previousVixAvg': vix_by_year.get('2024', {}).get('vixAvg'),
    'latestScatterKey': latest_monthly_point['key'] if latest_monthly_point else None,
    'riskIndicator': risk_indicator,
    'forecastChart': forecast_chart,
    'indexForecastChart': index_forecast_chart,
    'columns': columns,
    'riskScatter': monthly_scatter,
    'returns': [
        {
            'year': row['year'],
            'returnPct': round(row['returnPct'], 2),
            'bucket': row['bucket'],
            'annualized': bool(row.get('annualized', False)),
        }
        for row in full_range
    ],
}

ANNUAL_EXPORT.write_text(json.dumps(annual_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved', ANNUAL_EXPORT)
print('2024:', annual_payload['previousReturnPct'])
print('2025:', annual_payload['latestReturnPct'])



## Cell 4: Market map dataset

This section does **not download anything**.

It reads a **local manual source**:
- `sources/manual/notebook/sp500_market_map_returns.csv`

That file is treated as the notebook-driven source of truth for:
- company metadata
- sector and industry
- weights
- return horizons

The output is written to `datasets/exports/sp500-market-map.json`.


In [ ]:


def read_csv_rows(path: Path) -> list[dict[str, str]]:
    with path.open('r', encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))


def is_finite_positive(value: float) -> bool:
    return math.isfinite(value) and value > 0


def format_trillions(value: float) -> str:
    return f"${value / 1e12:.2f}T"


rows = read_csv_rows(MARKET_MAP_CSV)
companies = []
for row in rows:
    ticker = (row.get('ticker') or row.get('symbol') or '').strip()
    name = clean_company_name(row.get('display_name') or row.get('quote_name') or row.get('name') or row.get('company') or row.get('Security') or '')
    sector = (row.get('sector') or '').strip()
    industry = (row.get('industry') or row.get('subindustry') or row.get('sub-industry') or '').strip() or 'Unclassified'
    weight = parse_number(row.get('weight'))
    market_cap = parse_number(row.get('marketCap'))
    change = parse_number(row.get('change'))
    return_1d = parse_number(row.get('return_1d') or row.get('return1d'))
    return_ytd = parse_number(row.get('return_ytd') or row.get('returnYtd'))
    return_1y = parse_number(row.get('return_1y') or row.get('return1y'))
    return_5y = parse_number(row.get('return_5y') or row.get('return5y'))
    return_10y = parse_number(row.get('return_10y') or row.get('return10y'))
    if not ticker or not name or not sector:
        continue
    companies.append({
        'ticker': ticker,
        'name': name,
        'sector': sector,
        'industry': industry,
        'marketCap': market_cap if is_finite_positive(market_cap) else None,
        'weight': weight if math.isfinite(weight) and weight > 0 else None,
        'change': change if math.isfinite(change) else None,
        'return1d': return_1d if math.isfinite(return_1d) else None,
        'returnYtd': return_ytd if math.isfinite(return_ytd) else None,
        'return1y': return_1y if math.isfinite(return_1y) else None,
        'return5y': return_5y if math.isfinite(return_5y) else None,
        'return10y': return_10y if math.isfinite(return_10y) else None,
    })

if not any(company['weight'] is not None for company in companies):
    valid_market_caps = [company['marketCap'] for company in companies if is_finite_positive(company['marketCap'])]
    total_market_cap = sum(valid_market_caps)
    for company in companies:
        if is_finite_positive(company['marketCap']) and total_market_cap > 0:
            company['weight'] = company['marketCap'] / total_market_cap * 100

companies = [company for company in companies if company['weight'] is not None]
companies.sort(key=lambda company: company['weight'], reverse=True)

total_market_cap = sum(company['marketCap'] or 0 for company in companies)
sector_groups = defaultdict(list)
for company in companies:
    sector_groups[company['sector']].append(company)

sectors = []
for sector_name, members in sorted(sector_groups.items(), key=lambda item: sum(c['weight'] for c in item[1]), reverse=True):
    cfg = SECTOR_CONFIG.get(sector_name, {'key': re.sub(r'[^a-z0-9]+', '-', sector_name.lower()).strip('-'), 'es': sector_name, 'color': '#64748b'})
    sector_weight = round(sum(company['weight'] for company in members), 1)
    top = []
    for company in members:
        top.append({
            'ticker': company['ticker'],
            'label': company['ticker'],
            'name': company['name'],
            'industry': company['industry'],
            'weight': round(company['weight'], 4),
            'marketCap': company['marketCap'],
            'marketCapLabel': format_trillions(company['marketCap']) if is_finite_positive(company['marketCap'] or float('nan')) else None,
            'change': round(company['change'], 4) if company['change'] is not None else None,
            'return1d': round(company['return1d'], 4) if company['return1d'] is not None else None,
            'returnYtd': round(company['returnYtd'], 4) if company['returnYtd'] is not None else None,
            'return1y': round(company['return1y'], 4) if company['return1y'] is not None else None,
            'return5y': round(company['return5y'], 4) if company['return5y'] is not None else None,
            'return10y': round(company['return10y'], 4) if company['return10y'] is not None else None,
        })
    sectors.append({
        'key': cfg['key'],
        'name': {'en': sector_name, 'es': cfg['es']},
        'weight': sector_weight,
        'companies': len(members),
        'color': cfg['color'],
        'top': top,
    })

market_map_payload = {
    'generatedAt': __import__('datetime').datetime.utcnow().isoformat() + 'Z',
    'title': {'en': 'S&P 500 Market Map', 'es': 'Mapa del S&P 500'},
    'subtitle': {
        'en': 'Treemap-style composition and returns view built from the notebook CSV pipeline.',
        'es': 'Vista de composición y retornos tipo treemap construida desde el pipeline de notebook.',
    },
    'note': {
        'en': 'This export reads the notebook CSV source. If the CSV includes company weights they are used directly; otherwise weights are recomputed from market capitalization.',
        'es': 'Este export usa el CSV del notebook como fuente. Si el CSV trae pesos, se usan directamente; si no, los pesos se recalculan desde capitalización bursátil.',
    },
    'source': {
        'name': MARKET_MAP_CSV.name,
        'url': '',
    },
    'totalMarketCap': format_trillions(total_market_cap) if total_market_cap > 0 else None,
    'sectors': sectors,
}

MARKET_MAP_EXPORT.write_text(json.dumps(market_map_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved', MARKET_MAP_EXPORT)
print('Sectors:', len(sectors), '| Companies:', sum(len(sector['top']) for sector in sectors))



## Cell 5: Sector return series dataset

This section does **not download anything**.

It reads two **local manual sources**:
- the market-map CSV for current company weights
- the 15-year wide timeseries CSV for adjusted close history

The output is written to `datasets/exports/sp500-sector-returns.json`.


In [ ]:

weights_rows = read_csv_rows(MARKET_MAP_CSV)
price_rows = read_csv_rows(TIMESERIES_WIDE_CSV)

company_meta = []
for row in weights_rows:
    ticker = (row.get('ticker') or row.get('symbol') or '').strip()
    sector = (row.get('sector') or '').strip()
    weight = parse_number(row.get('weight'))
    if not ticker or not sector or not math.isfinite(weight) or weight <= 0:
        continue
    cfg = SECTOR_CONFIG.get(sector, {'key': re.sub(r'[^a-z0-9]+', '-', sector.lower()).strip('-'), 'es': sector, 'color': '#64748b'})
    company_meta.append({
        'ticker': ticker,
        'sector': sector,
        'sectorKey': cfg['key'],
        'sectorName': {'en': sector, 'es': cfg['es']},
        'color': cfg['color'],
        'weight': weight,
    })

sector_map = {}
for row in company_meta:
    bucket = sector_map.setdefault(row['sectorKey'], {
        'key': row['sectorKey'],
        'name': row['sectorName'],
        'color': row['color'],
        'totalWeight': 0.0,
        'members': [],
    })
    bucket['totalWeight'] += row['weight']
    bucket['members'].append({'ticker': row['ticker'], 'weight': row['weight']})

sectors = sorted(sector_map.values(), key=lambda sector: sector['totalWeight'], reverse=True)
series_state = {sector['key']: {'level': None, 'points': []} for sector in sectors}

for prev, current in zip(price_rows[:-1], price_rows[1:]):
    for sector in sectors:
        weighted_return = 0.0
        available_weight = 0.0
        for member in sector['members']:
            prev_price = parse_number(prev.get(member['ticker']))
            current_price = parse_number(current.get(member['ticker']))
            if not math.isfinite(prev_price) or not math.isfinite(current_price) or prev_price <= 0:
                continue
            daily_return = current_price / prev_price - 1
            weighted_return += member['weight'] * daily_return
            available_weight += member['weight']
        if available_weight <= 0:
            continue
        normalized_return = weighted_return / available_weight
        state = series_state[sector['key']]
        state['level'] = 100 if state['level'] is None else state['level'] * (1 + normalized_return)
        state['points'].append({
            'date': current['date'],
            'value': round(state['level'], 4),
            'coverage': round(available_weight / sector['totalWeight'], 4),
        })

sector_returns_payload = {
    'generatedAt': __import__('datetime').datetime.utcnow().isoformat() + 'Z',
    'source': {
        'weights': MARKET_MAP_CSV.name,
        'prices': TIMESERIES_WIDE_CSV.name,
    },
    'note': 'Sector series are chained from weighted daily constituent returns using current company weights normalized within each sector among available members.',
    'sectors': [],
}

for sector in sectors:
    sector_returns_payload['sectors'].append({
        'key': sector['key'],
        'name': sector['name'],
        'color': sector['color'],
        'weight': round(sector['totalWeight'], 4),
        'companies': len(sector['members']),
        'series': series_state[sector['key']]['points'],
    })

total_by_date = {}
for sector in sector_returns_payload['sectors']:
    for point in sector['series']:
        current = total_by_date.setdefault(point['date'], {'weighted': 0.0, 'weight': 0.0, 'coverage': 0.0})
        current['weighted'] += point['value'] * sector['weight']
        current['weight'] += sector['weight']
        current['coverage'] += point['coverage'] * sector['weight']

total_series = [
    {
        'date': date,
        'value': round(value['weighted'] / value['weight'], 4),
        'coverage': round(value['coverage'] / value['weight'], 4),
    }
    for date, value in sorted(total_by_date.items())
]

sector_returns_payload['sectors'].insert(0, {
    'key': 'total',
    'name': {'en': 'Total', 'es': 'Total'},
    'color': '#f3c557',
    'weight': 100,
    'companies': len(company_meta),
    'series': total_series,
})

SECTOR_RETURNS_EXPORT.write_text(json.dumps(sector_returns_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved', SECTOR_RETURNS_EXPORT)
print('Series:', len(sector_returns_payload['sectors']))



## Cell 6: Composition / bubbles dataset

This section **does not download anything if a local Visual Capitalist HTML snapshot exists**.

It first tries to read **local manual HTML** from:
- `composition/sources/raw/visualcapitalist-sp500-2026.html`
- `composition/sources/raw/visualcapitalist-sp500-2026-*.html`

If those files do not exist, it falls back to **downloading**:
- the open constituent list from GitHub
- a live S&P 500 weights table from `us500.com`

The output is written to `datasets/exports/sp500-bubbles.json`.


In [ ]:

WEIGHTS_URL = 'https://us500.com/tools/data/sp500-companies-by-weight'
CONSTITUENTS_URL = 'https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv'
VISUAL_CAPITALIST_URL = 'https://www.visualcapitalist.com/the-entire-sp-500-in-2026-in-one-chart/'

LAYOUT_CONFIG = {
    'Information Technology': {'x': 278, 'y': 760, 'r': 190},
    'Financials': {'x': 856, 'y': 430, 'r': 150},
    'Communication Services': {'x': 690, 'y': 165, 'r': 142},
    'Consumer Discretionary': {'x': 362, 'y': 155, 'r': 150},
    'Health Care': {'x': 720, 'y': 730, 'r': 126},
    'Industrials': {'x': 165, 'y': 433, 'r': 145},
    'Consumer Staples': {'x': 958, 'y': 163, 'r': 100},
    'Energy': {'x': 957, 'y': 826, 'r': 92},
    'Utilities': {'x': 544, 'y': 932, 'r': 74},
    'Materials': {'x': 86, 'y': 173, 'r': 78},
    'Real Estate': {'x': 896, 'y': 676, 'r': 74},
}


def decode_html(value: str) -> str:
    return (
        value.replace('&amp;', '&')
        .replace('&#8211;', '–').replace('&ndash;', '–')
        .replace('&#8217;', '’').replace('&rsquo;', '’')
        .replace('&#8220;', '"').replace('&#8221;', '"').replace('&quot;', '"')
    )


def normalize_vc_sector(sector: str) -> str:
    mapping = {
        'Info Tech': 'Information Technology',
        'Health Care': 'Health Care',
        'Communication Services': 'Communication Services',
        'Consumer Discretionary': 'Consumer Discretionary',
        'Consumer Staples': 'Consumer Staples',
        'Financials': 'Financials',
        'Industrials': 'Industrials',
        'Materials': 'Materials',
        'Energy': 'Energy',
        'Utilities': 'Utilities',
        'Real Estate': 'Real Estate',
    }
    return mapping.get(sector, sector)


def parse_visual_capitalist_rows(html: str) -> list[dict[str, Any]]:
    rows = []
    row_pattern = re.compile(r'<tr class="row-[^"]+">([\s\S]*?)</tr>')
    col_pattern = re.compile(r'<td class="column-(\d)[^"]*">([\s\S]*?)</td>')
    for row_html in row_pattern.findall(html):
        cells = {col: decode_html(value) for col, value in col_pattern.findall(row_html)}
        if cells.get('2') and cells.get('3') and cells.get('4'):
            rows.append({
                'name': clean_company_name(re.sub(r'<[^>]+>', ' ', cells['2']).replace('
', ' ').strip()),
                'sector': normalize_vc_sector(re.sub(r'<[^>]+>', ' ', cells['3']).strip()),
                'weight': float(cells['4'].replace('%', '').strip()),
            })
    return rows


def extract_weights_from_next_data(html: str) -> list[dict[str, Any]]:
    match = re.search(r'<script id="__NEXT_DATA__" type="application/json">([\s\S]*?)</script>', html)
    if not match:
        raise RuntimeError('Could not find __NEXT_DATA__ payload in the us500 page.')
    payload = json.loads(match.group(1))
    return payload['props']['pageProps']['data']


def build_bubbles_fallback() -> dict[str, Any]:
    constituents_csv = download_text(CONSTITUENTS_URL)
    weights_html = download_text(WEIGHTS_URL)
    constituents = list(csv.DictReader(constituents_csv.splitlines()))
    weight_rows = extract_weights_from_next_data(weights_html)
    constituent_map = {
        row['Symbol']: {
            'security': row['Security'],
            'sector': row['GICS Sector'],
        }
        for row in constituents
    }
    grouped = defaultdict(list)
    for item in weight_rows:
        meta = constituent_map.get(item['ticker'])
        if not meta or meta['sector'] not in SECTOR_CONFIG:
            continue
        grouped[meta['sector']].append({
            'name': clean_company_name(meta['security'] or item['name']),
            'label': item['ticker'],
            'weight': float(item['weight']),
        })
    sectors = []
    for sector_name in SECTOR_CONFIG:
        cfg = SECTOR_CONFIG[sector_name]
        layout = LAYOUT_CONFIG[sector_name]
        companies = sorted(grouped.get(sector_name, []), key=lambda row: row['weight'], reverse=True)
        sector_weight = round(sum(company['weight'] for company in companies), 1)
        if not companies:
            continue
        sectors.append({
            'key': cfg['key'],
            'name': {'en': sector_name, 'es': cfg['es']},
            'weight': sector_weight,
            'companies': len(companies),
            'color': cfg['color'],
            'x': layout['x'],
            'y': layout['y'],
            'r': layout['r'],
            'top': companies,
        })
    return {
        'generatedAt': __import__('datetime').datetime.utcnow().isoformat() + 'Z',
        'title': {'en': 'The Entire S&P 500', 'es': 'Todo el S&P 500'},
        'subtitle': {
            'en': 'Packed-bubbles style reconstruction using live S&P 500 constituent weights.',
            'es': 'Reconstrucción tipo packed bubbles usando pesos actuales del S&P 500.',
        },
        'note': {
            'en': 'This fallback uses us500.com weights and the open constituents list. Poster layout is still a reconstruction.',
            'es': 'Este fallback usa pesos de us500.com y el listado abierto de constituyentes. La composición sigue siendo una reconstrucción.',
        },
        'source': {
            'name': 'us500.com + datasets/s-and-p-500-companies',
            'url': WEIGHTS_URL,
        },
        'totalMarketCap': '$57.62T',
        'sectors': sectors,
    }


html_parts = []
full_html = VC_RAW_DIR / 'visualcapitalist-sp500-2026.html'
if full_html.exists():
    html_parts.append(full_html.read_text(encoding='utf-8'))
for fragment in sorted(VC_RAW_DIR.glob('visualcapitalist-sp500-2026-*.html')):
    html_parts.append(fragment.read_text(encoding='utf-8'))

if html_parts:
    vc_rows = parse_visual_capitalist_rows('
'.join(html_parts))
    grouped = defaultdict(list)
    for row in vc_rows:
        if row['sector'] not in SECTOR_CONFIG:
            continue
        grouped[row['sector']].append({'name': row['name'], 'label': row['name'], 'weight': row['weight']})
    sectors = []
    for sector_name in SECTOR_CONFIG:
        cfg = SECTOR_CONFIG[sector_name]
        layout = LAYOUT_CONFIG[sector_name]
        companies = sorted(grouped.get(sector_name, []), key=lambda row: row['weight'], reverse=True)
        sector_weight = round(sum(company['weight'] for company in companies), 1)
        if not companies:
            continue
        sectors.append({
            'key': cfg['key'],
            'name': {'en': sector_name, 'es': cfg['es']},
            'weight': sector_weight,
            'companies': len(companies),
            'color': cfg['color'],
            'x': layout['x'],
            'y': layout['y'],
            'r': layout['r'],
            'top': companies,
        })
    bubbles_payload = {
        'generatedAt': __import__('datetime').datetime.utcnow().isoformat() + 'Z',
        'title': {'en': 'The Entire S&P 500', 'es': 'Todo el S&P 500'},
        'subtitle': {
            'en': 'Packed-bubbles style reconstruction of the Visual Capitalist S&P 500 poster using the article table as source.',
            'es': 'Reconstrucción tipo packed bubbles del póster de Visual Capitalist usando la tabla del artículo como fuente.',
        },
        'note': {
            'en': 'This build uses the local Visual Capitalist article HTML as the source of truth for company, sector, and weight values.',
            'es': 'Este build usa el HTML local del artículo de Visual Capitalist como fuente base para empresa, sector y peso.',
        },
        'source': {
            'name': 'Visual Capitalist article table',
            'url': VISUAL_CAPITALIST_URL,
        },
        'totalMarketCap': '$57.62T',
        'sectors': sectors,
    }
else:
    bubbles_payload = build_bubbles_fallback()

BUBBLES_EXPORT.write_text(json.dumps(bubbles_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved', BUBBLES_EXPORT)
print('Sectors:', len(bubbles_payload['sectors']))



## Cell 7: Final summary

This cell does **not download anything**.

It only prints the output files generated by the notebook so the pipeline is easy to audit.


In [ ]:

print('Generated exports:')
for path in [ANNUAL_EXPORT, MARKET_MAP_EXPORT, SECTOR_RETURNS_EXPORT, BUBBLES_EXPORT]:
    print('-', path)
